# Preprocessing

In [26]:
import xarray as xr
import glob
files = glob.glob("your file path")
valid_files = []

#Check files one by one
for f in files:
    try:
        xr.open_dataset(f)
        valid_files.append(f)
    except Exception as e:
        print(f"Skip the bad file: {f}")
        print("Reason:", e)

print(f"Valid: {len(valid_files)}")

#Read_Combine_Select city
import xarray as xr

ds = xr.open_mfdataset(
    valid_files,
    combine="by_coords",
    parallel=False,
    chunks=None
)

ds_city = ds.sel(
    latitude=30.0,
    longitude=103.0,
    method="nearest"
)

#Select Pressure Layer
levels = [1000, 850, 500]

ds_sel = ds_city.sel(pressure_level=levels)


#Turn to Dataframe
#Note: Reset index here is used to turn '(valid_time, pressure_level)' these two indexes into normal columns in df,preparing for the following Pivot conversion
df = ds_sel.to_dataframe().reset_index()

#Data structure conversion,Pivot
df_wide = df.pivot_table(
    index="valid_time",
    columns="pressure_level",
    values=["t", "r", "u", "v"]
)

#Column index flattening

df_wide.columns = [
    f"{var}_{int(level)}"
    for var, level in df_wide.columns
]

#Reset index
df_wide = df_wide.reset_index()

#Make labels {delete}
import numpy as np

df_wide["hour"] = df_wide["valid_time"].dt.hour


#Note: Here we created artificial data labels to fill in the blanks. In actual engineering, you need to replace these with your own data labels
df_wide["Y"] = (
    0.3 * df_wide["t_850"]

    -0.2 * df_wide["r_850"]

    +0.05 * (df_wide["u_850"]**2 + df_wide["v_850"]**2)**0.5

    +3 * np.maximum(0, np.sin(2*np.pi*df_wide["hour"]/24))

    +2 * np.sin(2*np.pi*df_wide["valid_time"].dt.dayofyear/365)

    + np.random.normal(0, 0.5, len(df_wide))
)

#Save
df_wide.to_parquet("202305.parquet")
print('Successfully Saved')

Skip the bad file: Data/202305\20230502_12.nc
Reason: [Errno -101] NetCDF: HDF error: 'C:\\Users\\ASUS\\Desktop\\Carbon_Cycle\\Data\\202305\\20230502_12.nc'
Valid: 743
Successfully Saved


# Check The Result

In [29]:
import pandas as pd
pd.read_parquet("the name of you processed data file")

,valid_time,r_500,r_850,r_1000,t_500,t_850,t_1000,u_500,u_850,u_1000,v_500,v_850,v_1000,hour,Y
0,2023-05-01 00:00:00,44.735497,66.911285,83.428360,265.537109,286.138916,294.977539,9.661301,-1.080170,0.219482,-5.268723,-0.260513,-0.188766,0,80.478461
1,2023-05-01 01:00:00,42.455204,69.666382,71.021996,265.759033,286.088135,295.091797,8.391159,-0.497894,0.263443,-6.206924,-0.365082,-0.474106,1,72.191694
2,2023-05-01 02:00:00,41.550884,67.334824,62.532230,266.028076,286.451172,295.940186,8.363495,-0.328644,-0.096893,-5.296005,-0.877258,-0.685776,2,79.124781
3,2023-05-01 03:00:00,39.157047,66.939301,58.522205,266.694092,286.857910,296.635742,8.783508,-1.005981,-0.654877,-4.433411,-0.781570,-0.407043,3,73.001545
4,2023-05-01 04:00:00,40.279919,65.173943,53.186249,267.106934,287.581055,297.857666,9.305908,-1.472595,-0.990952,-4.012009,0.163589,-0.013000,4,77.370044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
738,2023-05-31 19:00:00,103.270920,70.760117,81.002754,270.842285,289.230957,297.573242,6.331207,1.201477,0.383316,4.093048,-0.099304,1.242401,19,77.529329
739,2023-05-31 20:00:00,97.907303,70.958733,79.945755,270.914795,289.470947,297.563477,4.938965,0.447800,-0.129730,2.400360,0.376694,1.202698,20,78.279382
740,2023-05-31 21:00:00,83.775101,73.019928,79.825409,271.321777,289.184082,297.544678,3.501358,-0.512619,-0.434372,0.880219,1.508026,1.149185,21,79.148876
741,2023-05-31 22:00:00,59.311584,72.944656,82.088913,271.974609,289.146484,297.669678,1.847977,-2.524185,-1.098984,0.277481,1.406540,0.189743,22,74.227956


# Check null values

In [30]:
df.isnull().sum()

valid_time        0
pressure_level    0
d                 0
cc                0
z                 0
o3                0
pv                0
r                 0
ciwc              0
clwc              0
q                 0
crwc              0
cswc              0
t                 0
u                 0
v                 0
w                 0
vo                0
number            0
latitude          0
longitude         0
expver            0
dtype: int64

**Note**: If there is any null values you should consider which method of filling missing values would be better for your project and implement it to your data.